# 01 — Data Prep

Load `Project Data.xlsx`, clean the two sheets (FULL DATA = 5 assets, FACTORS = 27 factors), align on a common monthly date index, build an equal-weight portfolio (EWP) as a 6th "portfolio," and persist clean copies for the other notebooks to reuse.

Data range:
- Assets: 2004-01-30 → 2013-12-31 (120 monthly returns, no NAs)
- Factors: 2004-01-30 → 2018-12-31 (180 monthly returns, no NAs)

We keep only the overlapping 120-month window for regression work; the full factor series is retained separately for any out-of-sample / scenario testing later.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

PROJECT_ROOT = Path('..').resolve()
DATA_XLSX   = PROJECT_ROOT.parent / 'Project Data.xlsx'   # PM Final Project/Project Data.xlsx
OUT_DIR     = PROJECT_ROOT / 'data'
OUT_DIR.mkdir(exist_ok=True)
print('Reading:', DATA_XLSX)

Reading: /sessions/brave-sharp-curie/mnt/Portfolio Management/PM Final Project/Project Data.xlsx


## Read FULL DATA (assets)

In [2]:
raw = pd.read_excel(DATA_XLSX, sheet_name='FULL DATA', header=None)

# Row 2 holds the asset labels (columns 1..5). Returns begin on row 3.
asset_labels = raw.iloc[2, 1:6].tolist()
assets = raw.iloc[3:, :6].copy()
assets.columns = ['Date'] + asset_labels
assets['Date'] = pd.to_datetime(assets['Date'])
assets = (assets.set_index('Date')
                 .apply(pd.to_numeric, errors='coerce')
                 .sort_index())
print('Assets shape:', assets.shape)
print('Date range  :', assets.index.min().date(), '→', assets.index.max().date())
assets.head()

Assets shape: (120, 5)
Date range  : 2004-01-30 → 2013-12-31


,Asset 1,Asset 2,Asset 3,Asset 4,Asset 5
Date,,,,,
2004-01-30,0.0063,0.0348,0.0326,0.0065,0.0118
2004-02-27,0.0084,-0.0325,0.0840,0.0200,0.0238
2004-03-31,0.0044,-0.0267,-0.0036,0.0112,0.0165
2004-04-30,-0.0178,-0.0467,-0.0855,-0.0284,-0.0439
2004-05-31,-0.0022,0.0496,0.0437,-0.0069,0.0153


## Read FACTORS

In [3]:
rawf = pd.read_excel(DATA_XLSX, sheet_name='FACTORS', header=None)
# Row 0: display names, Row 1: Bloomberg ticker, Row 2: 'Factor' tag, Row 3+: data
factor_names = rawf.iloc[0, 1:].tolist()
factors = rawf.iloc[3:, :].copy()
factors.columns = ['Date'] + factor_names
factors['Date'] = pd.to_datetime(factors['Date'])
factors = (factors.set_index('Date')
                    .apply(pd.to_numeric, errors='coerce')
                    .sort_index())
print('Factors shape:', factors.shape)
print('Date range  :', factors.index.min().date(), '→', factors.index.max().date())
factors.head()

Factors shape: (180, 27)
Date range  : 2004-01-30 → 2018-12-31


,US Muni,US Tbill 1-3yr,US AGG,Global AGG,US Dollar,Commodities,Long Short HF,Event Driven HF,HF FOF Div,Relative Value HF,...,MSCI Europe,MSCI Japan,MSCI Asia ex-J,US Growth,US Value,US Small Cap,US Mid Cap,Gold,S&P500,Cash
Date,,,,,,,,,,,,,,,,,,,,,
2004-01-30,0.0045,0.0021,0.0080,0.0037,-0.0206,0.0181,0.0195,0.0280,0.0154,0.0121,...,0.0117,0.0181,0.0602,0.0204,0.0176,0.0434,0.0291,-0.0313,0.0184,0.0007
2004-02-27,0.0145,0.0050,0.0108,0.0052,0.0056,0.0649,0.0111,0.0120,0.0109,0.0055,...,0.0293,-0.0029,0.0335,0.0064,0.0214,0.0090,0.0215,-0.0157,0.0139,0.0007
2004-03-31,-0.0048,0.0032,0.0075,0.0111,0.0182,0.0309,0.0036,0.0006,0.0044,0.0043,...,-0.0312,0.1343,-0.0171,-0.0186,-0.0088,0.0093,0.0002,0.0765,-0.0151,0.0008
2004-04-30,-0.0209,-0.0101,-0.0260,-0.0366,0.0114,-0.0177,-0.0208,-0.0056,-0.0071,-0.0051,...,-0.0083,-0.0542,-0.0550,-0.0116,-0.0244,-0.0510,-0.0367,-0.0931,-0.0157,0.0008
2004-05-31,-0.0023,-0.0011,-0.0040,0.0049,0.0170,0.0170,-0.0019,-0.0031,-0.0088,-0.0034,...,0.0162,-0.0351,-0.0336,0.0186,0.0102,0.0159,0.0248,0.0228,0.0137,0.0008


## Align on common dates + build EWP

- Keep the intersection of both date indices (2004-01 → 2013-12 → 120 months).
- Compute the equal-weight portfolio of the 5 assets (monthly rebalanced; simple arithmetic mean of monthly returns is consistent with EW monthly rebal).
- Build an excess-return version of every series by subtracting the Cash factor — many regression tasks prefer excess returns.

In [4]:
common_idx = assets.index.intersection(factors.index)
assets_a  = assets.loc[common_idx].copy()
factors_a = factors.loc[common_idx].copy()

# Equal-weight portfolio: monthly rebalanced arithmetic mean of asset returns
assets_a['EWP'] = assets_a[asset_labels].mean(axis=1)
print('Aligned window:', common_idx.min().date(), '→', common_idx.max().date(), '|', len(common_idx), 'months')
assets_a.describe().T[['mean','std','min','max']]

Aligned window: 2004-01-30 → 2013-12-31 | 120 months


,mean,std,min,max
Asset 1,0.0038,0.0077,-0.0178,0.0393
Asset 2,0.0070,0.0538,-0.1779,0.1214
Asset 3,0.0078,0.0721,-0.2688,0.2218
Asset 4,0.0036,0.0140,-0.0284,0.0422
Asset 5,0.0045,0.0206,-0.0943,0.0721
EWP,0.0053,0.0242,-0.1075,0.0624


In [5]:
# Excess returns (subtract Cash). Keep the non-excess versions too — useful for style analysis.
cash = factors_a['Cash']
assets_ex   = assets_a.subtract(cash, axis=0)
factors_ex  = factors_a.drop(columns=['Cash']).subtract(cash, axis=0)
print('Excess returns built. Factor count (ex-cash):', factors_ex.shape[1])

Excess returns built. Factor count (ex-cash): 26


## Quick sanity check — annualized risk/return of raw assets and EWP

In [6]:
ann = pd.DataFrame({
    'Ann Return': assets_a.mean() * 12,
    'Ann Vol'   : assets_a.std()  * np.sqrt(12),
})
ann['Sharpe (vs Cash)'] = (assets_a.subtract(cash, axis=0).mean() * 12) / (assets_a.std() * np.sqrt(12))
ann

,Ann Return,Ann Vol,Sharpe (vs Cash)
Asset 1,0.0455,0.0265,1.1203
Asset 2,0.0840,0.1863,0.3662
Asset 3,0.0933,0.2496,0.3104
Asset 4,0.0427,0.0486,0.5530
Asset 5,0.0544,0.0713,0.5409
EWP,0.0640,0.0840,0.5735


## Persist clean dataframes for downstream notebooks

In [7]:
assets_a.to_csv(OUT_DIR / 'assets_raw.csv')
factors_a.to_csv(OUT_DIR / 'factors_raw.csv')
assets_ex.to_csv(OUT_DIR / 'assets_excess.csv')
factors_ex.to_csv(OUT_DIR / 'factors_excess.csv')
# Full (unaligned) factor history retained for any out-of-sample testing later.
factors.to_csv(OUT_DIR / 'factors_full.csv')
print('Wrote:')
for p in sorted(OUT_DIR.iterdir()):
    print(' ', p.name)

Wrote:
  assets_excess.csv
  assets_raw.csv
  factors_excess.csv
  factors_full.csv
  factors_raw.csv


**Output contract for downstream notebooks**
- `data/assets_raw.csv` — 120×6 (A1..A5, EWP)
- `data/factors_raw.csv` — 120×27
- `data/assets_excess.csv` — 120×6, excess of Cash
- `data/factors_excess.csv` — 120×26, excess of Cash (Cash dropped)
- `data/factors_full.csv` — 180×27 full history (Cash retained), for later OOS tests